# OpenPlaque — ultra-low-memory left-main-only gate

This notebook deliberately **does not load the full source CCTA** and does not attempt LAD/LCX branches. It uses the cached ~1 mm source-evidence volume plus the frozen RCA centerline to answer one question first: can we identify a credible left-main trunk?


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_LEFT_OSTIUM = True
REUSE_LEFT_MAIN = True


## Step 3 — Install this branch


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch left-main-bifurcation-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install numpy pandas scipy scikit-image matplotlib psutil
import sys, os, gc, psutil
sys.path.insert(0, '/content/OpenPlaque/src')
def ram(label='RAM'):
    p=psutil.Process(os.getpid())
    print(f'{label}: RSS {p.memory_info().rss/1024**3:.2f} GB')
ram('After install')


## Step 4 — Load only the cached ~1 mm evidence and RCA reference
No DICOM extraction, SimpleITK source volume, nnU-Net, or full-resolution CCTA array is loaded in this notebook.


In [ ]:
from openplaque.left_main_low_memory_workflow import LeftMainLowMemoryWorkflow, ALGORITHM_VERSION
wf = LeftMainLowMemoryWorkflow('/content/drive/MyDrive/OpenPlaque', reuse=(REUSE_LEFT_OSTIUM and REUSE_LEFT_MAIN))
rca = wf.load_cached_inputs()
print('Algorithm:', ALGORITHM_VERSION)
print('1-mm RCA calibration:')
display(rca)
gc.collect(); ram('After cached inputs')


## Step 5 — Sparse left-ostium search
This uses only a thin aortic-wall candidate set; there is no full-volume coordinate grid.


In [ ]:
ostia = wf.find_ostia()
display(ostia.head(12))
gc.collect(); ram('After ostium search')


## Step 6 — Short local left-main search
Only 6–28 mm routes are considered, with a local graph box. LAD/LCX are intentionally deferred until the trunk itself is credible.


In [ ]:
best = wf.find_left_main()
print('Preliminary low-resolution left-main summary:')
display(best['summary'])
display(wf.candidates.head(15))
gc.collect(); ram('After left-main search')


## Step 7 — Create compact QC figures
The cross-section figure compares the proposed trunk with the frozen RCA, both sampled from the same cached ~1 mm evidence.


In [ ]:
figs = wf.make_figures()
for f in figs: print('Saved:', f)
gc.collect(); ram('After figures')


## Step 8 — Package report


In [ ]:
zip_path = wf.package()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LEFT_MAIN_LOW_MEMORY_REPORT_BACK.zip')


When finished, return to ChatGPT and say **Retrieve and analyze**. If this low-resolution gate looks credible, the next run will be a separate high-resolution QC that loads the source CCTA without keeping the graph/evidence arrays in memory.
